# ***Projeto Oraculum***: Análise de Viabilidade de Terreno

Este notebook contém o fluxo de análise do projeto Oraculum. O objetivo é processar imagens de satélite e de relevo para gerar um score de viabilidade para construção civil.

## 1. Configuração do Ambiente

 1. Abra o Terminal (ALT + F12 no PyCharm) e execute '***pip install -r requirements.txt***'
 2. Inicie a autenticação com o projeto no Google Cloud

> (Caso o terminal dê errado, execute a célula abaixo)

In [ ]:
'''Caso dê errado ou preguiça:'''
#Instalação de dependências
!pip install numpy opencv-python matplotlib tifffile imagecodecs rasterio earthengine-api geemap ipywidgets ipympl notebook

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, LinearSegmentedColormap
import os
import time
from datetime import datetime
import re
import ipywidgets
from ipywidgets import interact, IntSlider, FloatSlider, VBox, RadioButtons

# Biblioteca principal para interagir com o Google Earth Engine
import ee
import geemap

# Bibliotecas especializadas para arquivos GeoTIFF
import tifffile
import rasterio
from rasterio.warp import reproject, Resampling

#Nossas próprias bibliotecas
import processamento
import utilitarios
import obter_dados

In [ ]:
ID_PROJETO_GOOGLE = 'oraculum-eseg'
obter_dados.autenticar_ee(ID_PROJETO_GOOGLE)

## 2. Definição da Área e Aquisição de Dados

Aqui definimos o ponto central e o tamanho da área que desejamos analisar. As funções do módulo `utilitarios` convertem as coordenadas para o formato correto e criam um polígono da área de interesse (AOI). Em seguida, o módulo `obter_dados` utiliza essa AOI para baixar os dados de satélite e relevo do Google Earth Engine.

In [ ]:
# --- 1. DEFINA O LOCAL E TAMANHO ---
lat_dms = "20°14'58.2\"S"
lon_dms = "46°59'53.7\"W"
tamanho_km = 10

# --- 2. CONVERTER E CRIAR A ÁREA ---
lat_dd = utilitarios.dms_para_dd(lat_dms)
lon_dd = utilitarios.dms_para_dd(lon_dms)
coords_poligono = utilitarios.criar_bounding_box(lat_dd, lon_dd, tamanho_km)
area_de_interesse = ee.Geometry.Polygon(coords_poligono)

# --- 3. DEFINIR PASTA DE SAÍDA E BAIXAR OS DADOS ---
# A função agora retorna o caminho da pasta onde os arquivos foram salvos.
caminho_arquivo_satelite, caminho_arquivo_relevo = obter_dados.baixar_dados_da_area(
    area_de_interesse,
    pasta_mae="outputs"
)

print(f"\nArquivos para esta análise:")
print(f"Satélite: {caminho_arquivo_satelite}")
print(f"Relevo: {caminho_arquivo_relevo}")

In [ ]:
# --- CONFIGURAÇÃO DA ANÁLISE ---
# Para analisar um conjunto de dados, cole o timestamp da pasta aqui.
# Se você acabou de baixar, use o timestamp que apareceu na saída da célula anterior.
# Se quer analisar dados antigos, copie o nome da pasta de 'outputs'.

timestamp_da_analise = "20251118_091232" # ⇽ ÚNICO LUGAR PARA MUDAR

# --- Construção automática dos caminhos ---
pasta_da_sessao = os.path.join("outputs", timestamp_da_analise)
caminho_arquivo_satelite = os.path.join(pasta_da_sessao, f"{timestamp_da_analise}_satelite.tif")
caminho_arquivo_relevo = os.path.join(pasta_da_sessao, f"{timestamp_da_analise}_relevo.tif")

print(f"Pronto para analisar a sessão: {timestamp_da_analise}")
print(f"Verificando se os arquivos existem...")

# Checagem de segurança
if os.path.exists(caminho_arquivo_satelite) and os.path.exists(caminho_arquivo_relevo):
    print(">>> SUCESSO! Arquivos encontrados. Pode prosseguir com as próximas células de análise.")
else:
    print(">>> ERRO! Um ou ambos os arquivos não foram encontrados. Verifique o timestamp ou se você baixou o arquivo do Google Drive.")

## 3. Carga e Alinhamento de Dados

In [ ]:
# --- Célula de Carga e Alinhamento (Versão Final Sincronizada) ---

# A função agora retorna 5 produtos, que armazenamos em suas respectivas variáveis
(
    IMG_SATELITE,
    IMG_RELEVO,
    BANDA_VERMELHA_BRUTA,
    BANDA_VERDE_BRUTA,
    BANDA_NIR_BRUTA,
    nodata_relevo,
    RESOLUCAO_METROS
) = processamento.alinhar_imagens(
    caminho_arquivo_satelite,
    caminho_arquivo_relevo
)

with rasterio.open(caminho_arquivo_relevo) as src_rel:
    nodata_relevo = src_rel.nodata if src_rel.nodata is not None else IMG_RELEVO.min()

print("\nDados carregados e prontos para análise:")
print(f"- Imagem Visual (RGB): {IMG_SATELITE.shape}")
print(f"- Mapa de Relevo: {IMG_RELEVO.shape}")
print(f"- Banda Vermelha (Bruta): {BANDA_VERMELHA_BRUTA.shape}")
print(f"- Banda Verde (Bruta): {BANDA_VERDE_BRUTA.shape}")
print(f"- Banda NIR (Bruta): {BANDA_NIR_BRUTA.shape}")
print(f"- Resolução (metros): {RESOLUCAO_METROS} metros")


# --- VISUALIZAÇÃO DE CONFIRMAÇÃO ---
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Alinhamento de Imagens (RGB x Relevo)', fontsize=16)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title(f'Satélite Alinhado (RGB)')
ax[0].axis('off')

# Lógica para visualização correta do relevo
dados_validos_relevo = IMG_RELEVO != nodata_relevo
min_relevo = IMG_RELEVO[dados_validos_relevo].min()
max_relevo = IMG_RELEVO[dados_validos_relevo].max()
ax[1].imshow(IMG_RELEVO, cmap='viridis', vmin=min_relevo, vmax=max_relevo)
ax[1].set_title(f'Relevo Alinhado (Visual)')
ax[1].axis('off')

nome_arquivo_saida = os.path.join(pasta_da_sessao, 'comparacao_alinhamento.png')
plt.savefig(nome_arquivo_saida, dpi=300, bbox_inches='tight', pad_inches=0.1)

print(f"\n✔️ Imagem de comparação do alinhamento salva em: {nome_arquivo_saida}")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# --- Célula de Teste e Salvamento da Visualização V1 (Obsoleta) ---

print("Gerando e salvando a visualização 3D (v1)...")

# Chama a função v1 (obsoleta) que mantivemos para registro histórico
vis_v1 = processamento.fundir_imagens_v1(IMG_SATELITE, IMG_RELEVO)

# --- Exibir e Salvar ---
plt.figure(figsize=(12, 12))
plt.imshow(vis_v1)
plt.title('Visualização 3D - V1 (Método Simples)')
plt.axis('off')

# Constrói o caminho de saída DENTRO da pasta da sessão
# A variável 'pasta_da_sessao' foi criada na célula de aquisição de dados
nome_arquivo_saida = os.path.join(pasta_da_sessao, 'visualizacao_3D_v1.png')

# Salva a figura em alta qualidade
plt.savefig(nome_arquivo_saida, dpi=300, bbox_inches='tight', pad_inches=0.1)

print(f"Imagem da visualização V1 salva em: {nome_arquivo_saida}")

# Mostra a figura no notebook
plt.show()

In [ ]:
# --- Célula de Teste e Salvamento da Visualização V2 (Hillshade) ---

print("--- Testando e salvando a função de visualização 3D definitiva (Hillshade) ---")

# Chama a nova função final que usa Hillshade
imagem_3d_final = processamento.fundir_imagens_v2_3D(IMG_SATELITE, IMG_RELEVO)

# --- Exibir e Salvar ---
plt.figure(figsize=(12, 12))
plt.imshow(imagem_3d_final)
plt.title('Visualização 3D Final (Hillshade + HSV)')
plt.axis('off')

# Constrói o caminho de saída DENTRO da pasta da sessão
# A variável 'pasta_da_sessao' foi criada na célula de aquisição de dados
nome_arquivo_saida = os.path.join(pasta_da_sessao, 'visualizacao_3D_v2.png')
plt.savefig(nome_arquivo_saida, dpi=300, bbox_inches='tight', pad_inches=0.1)

print(f"Imagem da visualização V2 (Hillshade + HSV) salva em: {nome_arquivo_saida}")

# Mostra a figura no notebook
plt.show()

## 4. Análise de Hidrografia

In [ ]:
# --- ETAPA 2.1: Análise de Hidrografia (Comparativo V1 vs V2) ---

# Chama ambas as funções de processamento para gerar os mapas
MAPA_AGUA_V1 = processamento.criar_mapa_agua_v1(IMG_SATELITE)
MAPA_AGUA_V2 = processamento.criar_mapa_agua_v2(IMG_SATELITE)

# Define qual versão será a "oficial" para as próximas etapas do projeto
MAPA_AGUA = MAPA_AGUA_V2
print("Mapas de água V1 e V2 criados. A versão V2 foi definida como a oficial (MAPA_AGUA).")

# --- VISUALIZAÇÃO COMPARATIVA ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Comparativo dos Métodos de Detecção de Água', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL')
ax[0].axis('off')

ax[1].imshow(MAPA_AGUA_V1, cmap='gray')
ax[1].set_title('V1 - Método HSV')
ax[1].axis('off')

ax[2].imshow(MAPA_AGUA_V2, cmap='gray')
ax[2].set_title('V2 - Método RGB')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- SALVANDO OS RESULTADOS ---
caminho_v1 = os.path.join(pasta_da_sessao, 'mapa_agua_v1.png')
caminho_v2 = os.path.join(pasta_da_sessao, 'mapa_agua_v2.png')
caminho_comparacao = os.path.join(pasta_da_sessao, 'comparativo_hidrografia_v1_v2.png')
plt.imsave(caminho_v1, MAPA_AGUA_V1, cmap='gray')
plt.imsave(caminho_v2, MAPA_AGUA_V2, cmap='gray')
fig.savefig(caminho_comparacao, bbox_inches='tight')
print(f"Resultados da análise de hidrografia salvos na pasta da sessão.")

In [ ]:
def visualizar_mapa_agua_ajustavel(max_intensidade, min_dif_azul, min_dif_verde):
    # Chama a nossa nova função v3 com os valores dos sliders
    mapa_gerado = processamento.criar_mapa_agua_v3_ajustavel(
        IMG_SATELITE,
        limiar_intensidade=max_intensidade,
        diferenca_azul=min_dif_azul,
        diferenca_verde=min_dif_verde
    )

    # Plota o resultado
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(IMG_SATELITE)
    ax[0].set_title('Original')
    ax[0].axis('off')

    ax[1].imshow(mapa_gerado, cmap='gray')
    ax[1].set_title('Mapa de Água Resultante')
    ax[1].axis('off')

    plt.show()

# Cria o widget interativo que chama a função de visualização
interact(
    visualizar_mapa_agua_ajustavel,
    max_intensidade=IntSlider(value=100, min=50, max=150, step=5, description='Max Intensidade:'),
    min_dif_azul=IntSlider(value=15, min=0, max=50, step=1, description='Min Dif. Azul-Verm:'),
    min_dif_verde=IntSlider(value=5, min=0, max=50, step=1, description='Min Dif. Verde-Verm:')
);

In [ ]:
print("Análise de Hidrografia (NDWI com dados brutos)...")

# 1. Usa as bandas BRUTAS (BANDA_VERDE_BRUTA e BANDA_NIR_BRUTA) para o cálculo
banda_verde = BANDA_VERDE_BRUTA.astype(float)
banda_nir = BANDA_NIR_BRUTA.astype(float)

# 2. Calcula o NDWI
numerador = banda_verde - banda_nir
denominador = banda_verde + banda_nir
ndwi = np.divide(numerador, denominador, out=np.zeros_like(numerador), where=denominador!=0)

# 3. Cria a máscara de água com um limiar
limiar = 0.1
mascara_ndwi = ndwi > limiar
MAPA_AGUA = mascara_ndwi.astype(np.uint8) * 255
print("Mapa de Hidrografia (NDWI) criado com sucesso!")

# --- VISUALIZAÇÃO COMPLETA DO PROCESSO ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Processo de Análise de Hidrografia via NDWI', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL (RGB)')
ax[0].axis('off')

im = ax[1].imshow(ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
ax[1].set_title('MAPA DE NDWI (CIENTÍFICO)')
ax[1].axis('off')
fig.colorbar(im, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04)

ax[2].imshow(MAPA_AGUA, cmap='gray')
ax[2].set_title('MAPA DE ÁGUA FINAL (FEATURE 1)')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# --- SALVANDO O RESULTADO ---
caminho_saida_hidro = os.path.join(pasta_da_sessao, 'mapa_agua_v3.png')
caminho_saida_ndwi = os.path.join(pasta_da_sessao, 'mapa_ndwi_cientifico.png')
caminho_comparacao = os.path.join(pasta_da_sessao, 'comparativo_processo_ndwi.png')
plt.imsave(caminho_saida_hidro, MAPA_AGUA, cmap='gray')
plt.imsave(caminho_saida_ndwi, ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
fig.savefig(caminho_comparacao, bbox_inches='tight')
print(f"\n✔️ Mapa de hidrografia (NDWI) salvo em: {caminho_saida_hidro}")
print(f"✔️ Mapa NDWI científico salvo em: {caminho_saida_ndwi}")
print(f"✔️ Imagem de comparação do processo NDWI salva em: {caminho_comparacao}")

plt.show()

In [ ]:
caminho_saida_ndwi_calibracao = os.path.join(pasta_da_sessao, 'mapa_ndwi_escala_azuis.png')
plt.imsave(caminho_saida_ndwi_calibracao, ndwi, cmap='Blues', vmin=-0.5, vmax=1)
print(f"✔️ Mapa NDWI (base para calibração) salvo em: {caminho_saida_ndwi_calibracao}\n")

def visualizar_limiar_ndwi(limiar):
    mascara_ndwi = ndwi > limiar
    mapa_agua_gerado = mascara_ndwi.astype(np.uint8) * 255

    fig, ax = plt.subplots(1, 2, figsize=(16, 8))

    # Trocamos o mapa de cores para 'Blues', que é mais intuitivo para água.
    # Ajustamos vmin e vmax para focar nos valores positivos, onde a água está.
    im = ax[0].imshow(ndwi, cmap='Blues', vmin=-0.5, vmax=1)
    ax[0].set_title('MAPA DE NDWI (Escala de Azuis)')
    ax[0].axis('off')

    ax[1].imshow(mapa_agua_gerado, cmap='gray')
    ax[1].set_title(f'Resultado para Limiar = {limiar:.2f}')
    ax[1].axis('off')

    plt.show()

# Cria o widget interativo com um slider para o limiar
interact(
    visualizar_limiar_ndwi,
    limiar=FloatSlider(value=0.0, min=-0.2, max=0.4, step=0.01, description='Limiar NDWI:')
);

In [ ]:
# --- ETAPA 2.1: Análise de Hidrografia (Versão Modular) ---

# Chama a nossa nova função de processamento, usando as bandas brutas que já temos.
# Ela retorna tanto a máscara final quanto o mapa de NDWI para visualização.
MAPA_AGUA, ndwi_mapa = processamento.criar_mapa_agua_v4_ndwi(BANDA_VERDE_BRUTA, BANDA_NIR_BRUTA, limiar=0.0)


# --- VISUALIZAÇÃO COMPLETA DO PROCESSO ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Processo de Análise de Hidrografia via NDWI', fontsize=20)

ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL (RGB)')
ax[0].axis('off')

# Mostra o mapa de NDWI que a função retornou
im = ax[1].imshow(ndwi_mapa, cmap='RdYlBu', vmin=-1, vmax=1)
ax[1].set_title('MAPA DE NDWI (CIENTÍFICO)')
ax[1].axis('off')
fig.colorbar(im, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04)

# Mostra a máscara final que a função retornou
ax[2].imshow(MAPA_AGUA, cmap='gray')
ax[2].set_title('MAPA DE ÁGUA FINAL (FEATURE 1)')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# --- SALVANDO OS RESULTADOS ---
caminho_saida_mascara = os.path.join(pasta_da_sessao, 'mapa_agua_v3_calibrado.png')
caminho_saida_comparacao = os.path.join(pasta_da_sessao, 'comparacao_hidrografia_ndwi_final.png')

plt.imsave(caminho_saida_mascara, MAPA_AGUA, cmap='gray')
fig.savefig(caminho_saida_comparacao, dpi=300, bbox_inches='tight')
print(f"\nResultados da análise de hidrografia salvos na pasta da sessão.")

plt.show()

## 5. Análise de Declividade

In [ ]:
def visualizar_analise_relevo(limiar, alpha):

    # 1. Gera a máscara e o mapa de slope usando a função correta e os parâmetros dos sliders
    mapa_declividade_binario, mapa_declividade_graus = processamento.criar_mapa_declividade(
        IMG_RELEVO,
        resolucao_pixel=RESOLUCAO_METROS,
        nodata_value=nodata_relevo,
        limiar_graus=limiar
    )

    # Imprime a contagem de pixels para feedback
    total_pixels_ingremes = np.sum(mapa_declividade_binario > 0)
    print(f"Para o limiar {limiar:.1f}°, foram encontrados {total_pixels_ingremes} pixels íngremes.")

    # 2. Gera a imagem de overlay com a transparência (alpha) do slider
    imagem_overlay = processamento.criar_visualizacao_overlay(
        IMG_SATELITE,
        mapa_declividade_binario,
        cor_rgb=(255, 0, 0), # Vermelho
        alpha=alpha
    )

    # 3. Exibe a visualização final lado a lado
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))
    fig.suptitle(f'Análise de Declividade com Limiar > {limiar:.1f}°', fontsize=16)

    ax[0].imshow(IMG_SATELITE)
    ax[0].set_title('Imagem Original')
    ax[0].axis('off')

    ax[1].imshow(imagem_overlay)
    ax[1].set_title('Áreas Íngremes Destacadas')
    ax[1].axis('off')

    plt.show()

    # Guarda os resultados nas variáveis principais do notebook para uso futuro
    # (Atenção: estas variáveis serão atualizadas toda vez que você mexer no slider)
    globals()['MAPA_DECLIVIDADE'] = mapa_declividade_binario
    globals()['MAPA_DECLIVIDADE_GRAUS'] = mapa_declividade_graus


# Cria os sliders interativos
interact(
    visualizar_analise_relevo,
    limiar=FloatSlider(value=6.0, min=1.0, max=45.0, step=0.5, description='Limiar (Graus):'),
    alpha=FloatSlider(value=0.4, min=0.1, max=1.0, step=0.05, description='Transparência:')
);

In [ ]:
def visualizar_analise_combinada(limiar):

    # 1. Gera os dados para a imagem da DIREITA
    mapa_declividade_binario, mapa_indice_inclinacao = processamento.criar_mapa_declividade_visual(
        IMG_RELEVO,
        limiar_indice=limiar
    )

    # Imprime um feedback relevante para a nova imagem
    total_pixels = np.sum(mapa_declividade_binario > 0)
    print(f"Limiar de Índice: {limiar:.1f} -> Pixels íngremes: {total_pixels}")

    # 2. Exibe as visualizações lado a lado
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))
    fig.suptitle('Análise de Imagem de Satélite e Relevo', fontsize=16, y=0.93)

    # Gráfico da ESQUERDA: Imagem Original (do seu segundo script)
    ax[0].imshow(IMG_SATELITE)
    ax[0].set_title('Imagem Original')
    ax[0].axis('off')

    # Gráfico da DIREITA: Índice de Inclinação (do seu primeiro script)
    im = ax[1].imshow(mapa_indice_inclinacao + 0.1, cmap='magma', norm=LogNorm())
    ax[1].set_title(f'Índice de Inclinação (Visual)')
    ax[1].axis('off')

    plt.show()


# Cria o slider interativo, agora ajustado para o 'Limiar (Índice)'
interact(
    visualizar_analise_combinada,
    limiar=FloatSlider(value=50.0, min=1.0, max=250.0, step=1.0, description='Limiar (Índice):')
);

## 6. Análise de Vegetação

In [ ]:
# --- ETAPA 2.3: Análise de Vegetação ---

# Definimos o nosso critério: consideramos "vegetação densa" qualquer área
# com NDVI acima de 0.4. Este é um bom ponto de partida, mas pode ser ajustado.
LIMIAR_VEGETACAO = 0.4

# Chamamos a nossa nova função de processamento com as bandas brutas.
MAPA_NDVI, MAPA_VEGETACAO = processamento.criar_mapa_vegetacao_ndvi(
    BANDA_VERMELHA_BRUTA,
    BANDA_NIR_BRUTA,
    limiar=LIMIAR_VEGETACAO
)

# --- VISUALIZAÇÃO COMPARATIVA ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Análise de Vegetação via NDVI', fontsize=20)

# Imagem 1: A imagem de satélite original para referência
ax[0].imshow(IMG_SATELITE)
ax[0].set_title('IMAGEM ORIGINAL (RGB)')
ax[0].axis('off')

# Imagem 2: O mapa de NDVI "científico"
# Usamos um mapa de cores que vai do marrom (solo) ao verde (vegetação)
im_ndvi = ax[1].imshow(MAPA_NDVI, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
ax[1].set_title('MAPA DE NDVI (CIENTÍFICO)')
ax[1].axis('off')
fig.colorbar(im_ndvi, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04, label='Índice NDVI')

# Imagem 3: O nosso mapa final, a "feature" de vegetação
ax[2].imshow(MAPA_VEGETACAO, cmap='gray')
ax[2].set_title(f'VEGETAÇÃO DENSA (NDVI > {LIMIAR_VEGETACAO})')
ax[2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# --- SALVANDO OS RESULTADOS ---
caminho_saida_comparacao = os.path.join(pasta_da_sessao, 'comparacao_vegetacao.png')
fig.savefig(caminho_saida_comparacao, dpi=300, bbox_inches='tight')
print(f"\nFigura de comparação da vegetação salva em: {caminho_saida_comparacao}")

caminho_saida_vegetacao = os.path.join(pasta_da_sessao, 'mapa_vegetacao.png')
plt.imsave(caminho_saida_vegetacao, MAPA_VEGETACAO, cmap='gray')
print(f"Mapa de vegetação (binário) salvo em: {caminho_saida_vegetacao}")

plt.show()

## 7. Normalização das Feature

In [ ]:
# --- ETAPA 3.1.3: Normalização das Features --- (v1)
print("Normalizando as features para uma escala de pontuação universal...")
SCORE_DECLIVIDADE = processamento.normalizar_feature(MAPA_DECLIVIDADE, valor_positivo=1.0, valor_negativo=-1.0)
SCORE_VEGETACAO = processamento.normalizar_feature(MAPA_VEGETACAO, valor_positivo=1.0, valor_negativo=-1.0)
MODIFICADOR_AGUA = processamento.normalizar_feature(MAPA_AGUA, valor_positivo=0.0, valor_negativo=1.0)
print("Normalização concluída!")

# --- Correção Lógica: Neutralizar scores onde há água ---
SCORE_DECLIVIDADE[MAPA_AGUA == 255] = 0
SCORE_VEGETACAO[MAPA_AGUA == 255] = 0

# --- VISUALIZAÇÃO DA TRANSFORMAÇÃO (VERSÃO FINAL E CORRETA) ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Features Normalizadas para Análise de Viabilidade', fontsize=20)

# --- CORREÇÃO FINAL DO CMAP ---
# Usamos 'RdBu_r' (Revertido). Agora a convenção universal está correta:
# Vermelho = Ruim (valores negativos)
# Azul     = Bom  (valores positivos)
cmap_score = ('RdBu')

# Gráfico 1: Score Declividade
im_score = ax[0].imshow(SCORE_DECLIVIDADE, cmap=cmap_score, vmin=-1, vmax=1)
ax[0].set_title('SCORE DECLIVIDADE (Plano é Bom)')
ax[0].axis('off')

# Gráfico 2: Score Vegetação
ax[1].imshow(SCORE_VEGETACAO, cmap=cmap_score, vmin=-1, vmax=1)
ax[1].set_title('SCORE VEGETAÇÃO (Sem Mata é Bom)')
ax[1].axis('off')

# Gráfico 3: Modificador Água
ax[2].imshow(MODIFICADOR_AGUA, cmap='gray_r')
ax[2].set_title('MODIFICADOR ÁGUA')
ax[2].axis('off')

# Legenda (Colorbar) centralizada e correta
cbar_ax = fig.add_axes([0.3, 0.0, 0.4, 0.03])
fig.colorbar(im_score, cax=cbar_ax, orientation='horizontal', label='Pontuação de Viabilidade (-1 Ruim / +1 Bom)')

plt.subplots_adjust(top=0.9)
plt.show()

In [ ]:
# --- ETAPA 3.1.3: Normalização das Features --- (REVISADO)
print("Normalizando as features para uma escala de pontuação universal...")

# Criamos scores de -1 (ruim) a +1 (bom) para um cenário de CONSTRUÇÃO PADRÃO
SCORE_DECLIVIDADE = processamento.normalizar_feature(MAPA_DECLIVIDADE, valor_positivo=1.0, valor_negativo=-1.0)
SCORE_VEGETACAO = processamento.normalizar_feature(MAPA_VEGETACAO, valor_positivo=1.0, valor_negativo=-1.0)

# --- MUDANÇA PRINCIPAL AQUI ---
# A água agora também é um score, não um modificador.
# Terra (0) = +1.0 (bom)
# Água (255) = -1.0 (ruim)
SCORE_AGUA = processamento.normalizar_feature(MAPA_AGUA, valor_positivo=1.0, valor_negativo=-1.0)
print("Normalização concluída!")

# --- REMOVIDO ---
# As linhas abaixo que zeravam o score na água foram DELETADAS
# para permitir a análise de píeres/marinas.

# --- VISUALIZAÇÃO DA TRANSFORMAÇÃO ---
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Features Normalizadas para Análise de Viabilidade', fontsize=20)

cmap_score = 'RdBu' # Vermelho = Ruim (-1), Azul = Bom (+1)

# Gráfico 1: Score Declividade
im_score = ax[0].imshow(SCORE_DECLIVIDADE, cmap=cmap_score, vmin=-1, vmax=1)
ax[0].set_title('SCORE DECLIVIDADE (Plano é Bom)')
ax[0].axis('off')

# Gráfico 2: Score Vegetação
ax[1].imshow(SCORE_VEGETACAO, cmap=cmap_score, vmin=-1, vmax=1)
ax[1].set_title('SCORE VEGETAÇÃO (Sem Mata é Bom)')
ax[1].axis('off')

# Gráfico 3: Score Água (Atualizado)
ax[2].imshow(SCORE_AGUA, cmap=cmap_score, vmin=-1, vmax=1)
ax[2].set_title('SCORE ÁGUA')
ax[2].axis('off')

# Legenda (Colorbar) centralizada
cbar_ax = fig.add_axes([0.3, 0.0, 0.4, 0.03])
fig.colorbar(im_score, cax=cbar_ax, orientation='horizontal', label='Pontuação de Viabilidade (-1 Ruim / +1 Bom)')

plt.subplots_adjust(top=0.9)
plt.show()

## 8. Feature de Proximidade da Água

In [ ]:
# --- ETAPA 8: Feature de Proximidade da Água (VERSÃO INTERATIVA) ---

# 1. Pré-cálculo (A parte Lenta)
# Calculamos o mapa de distância, que é pesado e só precisa ser feito uma vez.
mapa_terra = (MAPA_AGUA == 0).astype(np.uint8)
MAPA_DISTANCIA = cv2.distanceTransform(mapa_terra, cv2.DIST_L2, 5)
print("Mapa de Distância da Água (Base) calculado. Pronto para calibrar.")

# 2. Esta função interna será chamada pelos sliders
def calibrar_score_proximidade(dist_ideal, dist_max):

    # Pega o mapa de distância que já calculamos
    mapa_distancia = MAPA_DISTANCIA.copy()

    # 2.A. Recalcula o score (A parte Rápida)
    # Esta é a fórmula que estava dentro do processamento.py
    score_prox = 1 - 2 * ((mapa_distancia - dist_ideal) / (dist_max - dist_ideal))

    # 2.B. Garante que o score fique entre -1 e 1 e zera na água.
    score_prox = np.clip(score_prox, -1.0, 1.0)
    score_prox[MAPA_AGUA == 255] = 0  # Zera o score NA água

    # --- ATUALIZA A VARIÁVEL GLOBAL ---
    # Isso é crucial para o Painel Final (Célula 9) usar o valor calibrado
    globals()['SCORE_PROX_AGUA'] = score_prox

    # --- VISUALIZAÇÃO (a mesma de antes) ---
    fig, ax = plt.subplots(1, 3, figsize=(24, 8))
    fig.suptitle('Calibração da Proximidade à Água', fontsize=20)

    # Gráfico 1: Mapa de Água
    ax[0].imshow(MAPA_AGUA, cmap='gray')
    ax[0].set_title('MAPA DE ÁGUA ORIGINAL')
    ax[0].axis('off')

    # Gráfico 2: Mapa de Distância (fixo)
    im_dist = ax[1].imshow(MAPA_DISTANCIA, cmap='viridis')
    ax[1].set_title('MAPA DE DISTÂNCIA (EM PIXELS)')
    ax[1].axis('off')
    fig.colorbar(im_dist, ax=ax[1], orientation='horizontal', fraction=0.046, pad=0.04, label='Distância em Pixels da Água')

    # Gráfico 3: O Score RESULTANTE
    score_visualizacao = score_prox.copy()
    mascara_agua = MAPA_AGUA == 255
    score_visualizacao[mascara_agua] = np.nan

    im_score = ax[2].imshow(score_visualizacao, cmap='RdBu', vmin=-1, vmax=1)
    ax[2].set_title(f'SCORE (Ideal: {dist_ideal}px, Max: {dist_max}px)')
    ax[2].axis('off')
    fig.colorbar(im_score, ax=ax[2], orientation='horizontal', fraction=0.046, pad=0.04, label='Score de Proximidade (-1 Longe, +1 Perto)')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# 3. Cria os Sliders
interact(
    calibrar_score_proximidade,
    dist_ideal=IntSlider(value=10, min=1, max=50, step=1, description='Dist. Ideal:'),
    dist_max=IntSlider(value=150, min=50, max=500, step=10, description='Dist. Máxima:')
);

## 9. Painel de Analises e Visualização

In [ ]:
# --- ETAPA 3.2: O Painel de Controle do Oraculum ---

# Esta função interna será chamada toda vez que um slider for movido.
def visualizar_score_final(peso_declividade, peso_vegetacao, peso_prox_agua):

    # 1. Normalização dos Pesos
    # Para garantir uma média ponderada justa, garantimos que a soma dos pesos seja 1 (ou 100%).
    soma_pesos = peso_declividade + peso_vegetacao + peso_prox_agua
    if soma_pesos == 0: # Evita divisão por zero se todos os sliders estiverem em 0
        soma_pesos = 1

    w_dec = peso_declividade / soma_pesos
    w_veg = peso_vegetacao / soma_pesos
    w_prox = peso_prox_agua / soma_pesos

    # 2. A Fórmula do Oraculum: Cálculo do Score Ponderado
    SCORE_FINAL = (
        (SCORE_DECLIVIDADE * w_dec) +
        (SCORE_VEGETACAO * w_veg) +
        (SCORE_PROX_AGUA * w_prox)
    )

    # 3. Aplica a máscara final para zerar o score na água
    SCORE_FINAL[MAPA_AGUA == 255] = np.nan # Usamos NaN para que o Matplotlib ignore a área

    # 4. Geração do Heatmap de Viabilidade
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Sobrepõe a imagem de satélite com baixa opacidade para dar contexto
    ax.imshow(IMG_SATELITE, alpha=0.6)

    # --- 4.1 GERAÇÃO DO HEATMAP (AQUI ESTÃO AS MUDANÇAS) ---

    # Paleta customizada com cores desaturadas
    colors = ['#A82223', '#F7F7F6', '#30503A'] # Vermelho Coral, Cinza Claro, Verde Coral
    custom_cmap = LinearSegmentedColormap.from_list('custom_calm', colors, N=256)
    # Desenha o heatmap do score por cima
    im = ax.imshow(SCORE_FINAL, cmap=custom_cmap, vmin=-1, vmax=1)


    ax.set_title('Heatmap de Viabilidade do Terreno', fontsize=16)
    ax.axis('off')

    # Adiciona a legenda (colorbar)
    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.02, fraction=0.04)
    cbar.set_label('Pontuação Final de Viabilidade (-1 Inviável / +1 Ideal)', fontsize=12)

    plt.show()


# Cria o widget interativo com os sliders para os pesos
interact(
    visualizar_score_final,
    peso_declividade=IntSlider(value=70, min=0, max=100, step=5, description='Peso Declividade:'),
    peso_vegetacao=IntSlider(value=20, min=0, max=100, step=5, description='Peso Vegetação:'),
    peso_prox_agua=IntSlider(value=10, min=0, max=100, step=5, description='Peso Prox. Água:')
);


In [ ]:
# --- ETAPA 4: O Painel de Controle do Oraculum (VERSÃO FINAL com RadioButtons) ---

# Criamos um dicionário para "traduzir" as opções de texto em números
mapeamento_agua = {
    'Ignorar (Ponte)': 100,
    'Evitar (Casa)': 0,
    'Procurar (Píer)': -100
}

def visualizar_score_final(peso_declividade, peso_vegetacao, peso_prox_agua, peso_agua_string):

    # 1. Traduz a string do RadioButton para o peso numérico
    peso_agua = mapeamento_agua[peso_agua_string]

    # 2. Normalização dos Pesos (usando valores absolutos)
    soma_pesos_abs = (
        abs(peso_declividade) + abs(peso_vegetacao) +
        abs(peso_prox_agua) + abs(peso_agua)
    )
    if soma_pesos_abs == 0:
        soma_pesos_abs = 1

    w_dec = peso_declividade / soma_pesos_abs
    w_veg = peso_vegetacao / soma_pesos_abs
    w_prox = peso_prox_agua / soma_pesos_abs
    w_agua = peso_agua / soma_pesos_abs

    # 3. A Fórmula do Oraculum
    SCORE_FINAL = (
        (SCORE_DECLIVIDADE * w_dec) +
        (SCORE_VEGETACAO * w_veg) +
        (SCORE_PROX_AGUA * w_prox) +
        (SCORE_AGUA * w_agua)
    )

    # 4. Geração do Heatmap
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(IMG_SATELITE, alpha=0.6)

    colors = ['#ff0000', '#F7F7F6', '#08f26e']
    custom_cmap = LinearSegmentedColormap.from_list('custom_calm', colors, N=256)

    score_visualizacao = SCORE_FINAL.copy()

    # Se o usuário NÃO quer procurar água (peso 0 ou 100), nós escondemos a água.
    if peso_agua >= 0:
         score_visualizacao[MAPA_AGUA == 255] = np.nan

    im = ax.imshow(score_visualizacao, cmap=custom_cmap, vmin=-1, vmax=1)

    ax.set_title('Heatmap de Viabilidade do Terreno', fontsize=16)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.02, fraction=0.04)
    cbar.set_label('Pontuação Final de Viabilidade (-1 Inviável / +1 Ideal)', fontsize=12)

    plt.show()


# --- ATUALIZAÇÃO DOS SLIDERS ---
# Trocamos o slider da água por RadioButtons
interact(
    visualizar_score_final,
    peso_declividade=IntSlider(value=70, min=0, max=100, step=10, description='Peso Declividade:'),
    peso_vegetacao=IntSlider(value=20, min=0, max=100, step=10, description='Peso Vegetação:'),
    peso_prox_agua=IntSlider(value=10, min=0, max=100, step=10, description='Peso Prox. Água:'),

    # --- MUDANÇA AQUI ---
    peso_agua_string=RadioButtons(
        options=['Evitar (Casa)', 'Ignorar (Ponte)', 'Procurar (Píer)'],
        value='Evitar (Casa)',
        description='Análise de Água:'
    )
);